In [1]:
import time; _t0 = time.time()

#basic imports
import os
import warnings
warnings.filterwarnings('ignore') 

#data manipulation
import pandas as pd 
import numpy as np


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Time series and statistics
# autocorrolation and partial autocorrelation
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# Seasonal trend decomposition using loss: break a time series in to the trend, seasonality, residuel components
from statsmodels.tsa.seasonal import STL

#Modeling
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.optimize import minimize

In [2]:
DATA_PATH = "data"

# Load datasets
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"), parse_dates=["date"])
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"), parse_dates=["date"])
oil = pd.read_csv(os.path.join(DATA_PATH, "oil.csv"), parse_dates=["date"])
holidays = pd.read_csv(os.path.join(DATA_PATH, "holidays_events.csv"), parse_dates=["date"])
stores = pd.read_csv(os.path.join(DATA_PATH, "stores.csv"))
transactions = pd.read_csv(os.path.join(DATA_PATH, "transactions.csv"), parse_dates=["date"])

# Quick check
print("DATA_PATH:", DATA_PATH)
print(f"train: {train.shape}, test: {test.shape}")

DATA_PATH: data
train: (3000888, 6), test: (28512, 5)


In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[us]
 2   store_nbr    int64         
 3   family       str           
 4   sales        float64       
 5   onpromotion  int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(1)
memory usage: 137.4 MB


In [4]:
train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [5]:
train.isnull().sum()

id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

In [6]:
print(test.shape)
test.info()

(28512, 5)
<class 'pandas.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           28512 non-null  int64         
 1   date         28512 non-null  datetime64[us]
 2   store_nbr    28512 non-null  int64         
 3   family       28512 non-null  str           
 4   onpromotion  28512 non-null  int64         
dtypes: datetime64[us](1), int64(3), str(1)
memory usage: 1.1 MB


In [ ]:
test.head()

id             0
date           0
store_nbr      0
family         0
onpromotion    0
dtype: int64

In [10]:
print(oil.shape)
oil.info()

(1218, 2)
<class 'pandas.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1218 non-null   datetime64[us]
 1   dcoilwtico  1175 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 19.2 KB


In [ ]:
oil.head()
oil.isnull().sum()
#there are some days that
#  oil price is missing, we can fill them with interpolation

date           0
dcoilwtico    43
dtype: int64

In [14]:
print(holidays.shape)
holidays.info()

(350, 6)
<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         350 non-null    datetime64[us]
 1   type         350 non-null    str           
 2   locale       350 non-null    str           
 3   locale_name  350 non-null    str           
 4   description  350 non-null    str           
 5   transferred  350 non-null    bool          
dtypes: bool(1), datetime64[us](1), str(4)
memory usage: 14.1 KB


In [16]:
holidays.head()
holidays.isnull().sum()

date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

In [17]:
print(stores.shape)
stores.info()

(54, 5)
<class 'pandas.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   store_nbr  54 non-null     int64
 1   city       54 non-null     str  
 2   state      54 non-null     str  
 3   type       54 non-null     str  
 4   cluster    54 non-null     int64
dtypes: int64(2), str(3)
memory usage: 2.2 KB


In [18]:
stores.isnull().sum()

store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

In [19]:
print(transactions.shape)
transactions.info()

(83488, 3)
<class 'pandas.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          83488 non-null  datetime64[us]
 1   store_nbr     83488 non-null  int64         
 2   transactions  83488 non-null  int64         
dtypes: datetime64[us](1), int64(2)
memory usage: 1.9 MB


In [21]:
transactions.head()
transactions.isnull().sum()

date            0
store_nbr       0
transactions    0
dtype: int64